# BistBull Phase 5 — BIST30 Non-Bank Recalibration

Phase 4.7 fits 5 sembol (n=155/metric) ile yapıldı. Phase 5 BIST30'un 26 non-bank sembolü ile yeniden kalibrasyon yapar (n≈800/metric — knot başına ~80 sample, 5 kat daha güvenli fit'ler).

Banka semboller (AKBNK, GARAN, ISCTR, YKBNK) Phase 6'da ayrı pipeline ile ele alınacak çünkü bilanço schema'ları farklı.

**Çıktı:** `reports/fa_isotonic_fits_v2.json` — production'da `?scoring_version=calibrated_2026Q2` query param ile etkinleşir.

**Beklenen süre:** ~3-4 saat (5 sembol için 30 dk → 26 sembol için ~3 saat).

In [ ]:
# ============================================================
# Setup — KERNEL RESTART sonrası bu hücreyi çalıştır
# ============================================================
import os, sys, glob, shutil

# 1. Zip auto-detect (Mac '(1).zip' ekleyebilir)
candidates = sorted(glob.glob('/content/bistbull*.zip'),
                    key=os.path.getmtime, reverse=True)
if not candidates:
    raise SystemExit('❌ /content/ altında bistbull*.zip yok — yükle')
zip_path = candidates[0]
print(f'✅ Zip: {zip_path}')

# Standardize
target = '/content/bistbull.zip'
if zip_path != target:
    shutil.copy(zip_path, target)

# 2. ENV — import'lardan ÖNCE
os.environ['BISTBULL_DB_PATH'] = '/content/bistbull.db'
os.environ['JWT_SECRET'] = 'colab-' + 'x' * 40
for p in glob.glob('/content/bistbull.db*'):
    os.remove(p)

# 3. Unzip
%cd /content
!rm -rf bistbull_repo
!unzip -q bistbull.zip -d bistbull_repo_tmp
# Find the actual repo dir inside
import os
inner = [d for d in os.listdir('/content/bistbull_repo_tmp')
         if os.path.isdir(f'/content/bistbull_repo_tmp/{d}')][0]
shutil.move(f'/content/bistbull_repo_tmp/{inner}', '/content/bistbull_repo')
shutil.rmtree('/content/bistbull_repo_tmp')
%cd /content/bistbull_repo

# 4. Phase 5 patch sanity
import subprocess
patch_check = subprocess.run(
    ['grep', '-c', 'CALIBRATED_V2_VERSION', 'engine/scoring_calibrated.py'],
    capture_output=True, text=True
)
if int(patch_check.stdout.strip() or 0) < 1:
    raise SystemExit('❌ Bu zip Phase 5 öncesi! Yeni zip kullan.')
print('✅ Phase 5 scaffold zip içinde')

# 5. Bağımlılıklar
!pip install -q borsapy pandas numpy 2>&1 | tail -2
!pip install -q --break-system-packages fastapi cachetools PyJWT argon2-cffi 2>&1 | tail -1

# 6. Path + DB
sys.path.insert(0, '/content/bistbull_repo')
import logging
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s %(levelname)s %(name)s: %(message)s')

from infra.storage import init_db, DB_PATH
init_db()
print(f'✅ DB initialized: {DB_PATH}')

# 7. Phase 5 universe
from config import UNIVERSE_BIST30_NON_BANK
print(f'\nPhase 5 target: {len(UNIVERSE_BIST30_NON_BANK)} non-bank symbols')
print(', '.join(UNIVERSE_BIST30_NON_BANK))

In [ ]:
# ============================================================
# AŞAMA A — Fiyat backfill (~30-40 dk for 26 symbols)
# ============================================================
from datetime import date
from research.ingest_prices import ingest
from config import UNIVERSE_BIST30_NON_BANK

result = ingest(
    symbols=UNIVERSE_BIST30_NON_BANK,
    from_date=date(2018, 4, 1),
    to_date=date(2026, 9, 30),
    dry_run=False,
    source=None,
    resume=True,  # Phase 5 retry-friendly
    max_workers=2,
)
print(f"\nPrice backfill: {result['totals']['symbols']} symbols, "
      f"{result['totals']['bars']} bars")
if result.get('errors'):
    print(f"⚠️ {len(result['errors'])} symbol error'i:")
    for sym, err in result['errors'].items():
        print(f"   {sym}: {err[:100]}")

# Sanity gate
import sqlite3
conn = sqlite3.connect(DB_PATH)
ok_syms = []
for sym in UNIVERSE_BIST30_NON_BANK:
    n = conn.execute('SELECT COUNT(*) FROM price_history_pit WHERE symbol=?',
                     (sym,)).fetchone()[0]
    status = '✅' if n >= 1500 else ('⚠️' if n >= 500 else '❌')
    if n >= 500:
        ok_syms.append(sym)
    print(f'  {status} {sym}: {n} bars')
conn.close()

print(f'\n✅ {len(ok_syms)}/{len(UNIVERSE_BIST30_NON_BANK)} symbols viable for Phase 5')
assert len(ok_syms) >= 20, '❌ Çok az sembolde yeterli veri var. Devam etme.'

In [ ]:
# ============================================================
# AŞAMA B — FA ingest (~2-3 saat)
# ============================================================
import os
os.makedirs('reports', exist_ok=True)

syms_csv = ','.join(ok_syms)
print(f'FA ingest: {len(ok_syms)} symbols')

!rm -f reports/fa_events_v2.csv reports/fa_events_v2_checkpoint.json
!python scripts/ingest_fa_for_calibration.py \
    --symbols={syms_csv} \
    --start=2018-01-01 --end=2026-04-01 \
    --out=reports/fa_events_v2.csv \
    --checkpoint=reports/fa_events_v2_checkpoint.json \
    --sleep-between-symbols=2.0 \
    --log-level=INFO 2>&1 | tail -50

print('\n=== fa_events_v2.csv ===')
!wc -l reports/fa_events_v2.csv
!awk -F, 'NR>1 {print $1}' reports/fa_events_v2.csv | sort | uniq -c | head -30

import subprocess
n = int(subprocess.check_output(['wc', '-l', 'reports/fa_events_v2.csv']).split()[0])
assert n > 1000, f'❌ Çok az event ({n} satır). Bana yaz.'

In [ ]:
# ============================================================
# AŞAMA C — Calibration v2 (~1 dk)
# ============================================================
!python scripts/calibrate_fa_from_events.py \
    --events=reports/fa_events_v2.csv \
    --out-fits=reports/fa_isotonic_fits_v2.json \
    --out-summary=reports/fa_calibration_summary_v2.md 2>&1 | tail -10

import json
with open('reports/fa_isotonic_fits_v2.json') as f:
    fits = json.load(f)
print(f'\n📊 v2 fits: {len(fits)} metric')
for k, fit in sorted(fits.items()):
    print(f"  {k:24} n={fit['n_samples']:4d}, knots={len(fit['x_knots']):2d}, "
          f"y∈[{fit['y_min']:+.3f}, {fit['y_max']:+.3f}]")

assert len(fits) >= 10, f'❌ Sadece {len(fits)} metric'

print('\n=== summary v2 ===')
!cat reports/fa_calibration_summary_v2.md

# Health check
!python scripts/check_calibration_health.py --strict --quiet 2>&1 | tail -3

In [ ]:
# ============================================================
# AŞAMA D — Q1 vs Q2 karşılaştırması
# ============================================================
import json

q1 = json.load(open('reports/fa_isotonic_fits.json'))
q2 = json.load(open('reports/fa_isotonic_fits_v2.json'))

print(f'Q1 fits: {len(q1)} metric')
print(f'Q2 fits: {len(q2)} metric')
print(f'Common: {len(set(q1) & set(q2))}')
print(f'Q1 only: {set(q1) - set(q2)}')
print(f'Q2 only: {set(q2) - set(q1)}')

print('\n--- Knot count + sample size karşılaştırması ---')
print(f'{"Metric":24} {"Q1 knots":>9} {"Q2 knots":>9} {"Q1 n":>6} {"Q2 n":>6} {"Δ knots":>8}')
for k in sorted(set(q1) | set(q2)):
    q1f = q1.get(k, {})
    q2f = q2.get(k, {})
    q1k = len(q1f.get('x_knots', []))
    q2k = len(q2f.get('x_knots', []))
    q1n = q1f.get('n_samples', '-')
    q2n = q2f.get('n_samples', '-')
    dk = q2k - q1k if (q1k and q2k) else '-'
    print(f'{k:24} {q1k:>9} {q2k:>9} {q1n:>6} {q2n:>6} {dk:>8}')

In [ ]:
# ============================================================
# AŞAMA E — Zip + indir
# ============================================================
!cd reports && zip -r /content/fa_results_v2.zip \
    fa_events_v2.csv fa_isotonic_fits_v2.json fa_calibration_summary_v2.md
!ls -la /content/fa_results_v2.zip

from google.colab import files
files.download('/content/fa_results_v2.zip')

print('\n✅ Phase 5 fits hazır. Bana yükle, commit ederim.')